# Basic analysis

Getting to know a session's data via `Mouse`/`Session`

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from CalciumKit.mouse import Mouse
from CalciumKit.trial_analysis import eta_all

sns.set_style("ticks")
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 1,
    "xtick.major.width": 1,
    "ytick.major.width": 1,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


In [ ]:
raw_root = Path(r"D:\ImagingData\Raw")
processed_root = Path(r"D:\ImagingData\Processed\Chandler")

mouse = Mouse("Chandler", raw_root, processed_root)
session = mouse.sessions["day2"]
session.task = "tfc"


## What's in this session

In [ ]:
for color_name, s2p in [("red", session.red), ("green", session.green)]:
    if s2p is None:
        print(f"{color_name}: not curated yet")
        continue
    print(f"{color_name}: {s2p.is_cell.sum()} / {len(s2p.is_cell)} ROIs are cells, {s2p.F.shape[1]} frames")

print("trial structure:", session.task)
print("tone times:", session.trial_structure.tone_times)
print("shock times:", session.trial_structure.shock_times)


## Shock-triggered average, both channels

Styled to match `shock_avg_pub` from the rotary notebooks.

In [ ]:
def plot_eta(traces, timestamps, event_times, color, ylabel, title, event_label="Shock", zscore=True):
    t_rel, mean_trace, (lo, hi), sig_runs, per_cell = eta_all(
        traces, timestamps, event_times, window_pre=5, window_post=15, zscore=zscore
    )

    fig, ax = plt.subplots(figsize=(2.5, 2), dpi=600)

    ax.fill_between(t_rel, lo, hi, color=color, alpha=0.25, linewidth=0, zorder=2)
    ax.plot(t_rel, mean_trace, color=color, linewidth=2, zorder=3)

    ax.axvline(0, color="black", linestyle="--", linewidth=1.2, zorder=4)
    ax.axhline(0, color="0.55", linestyle=":", linewidth=0.8, zorder=0)

    ylim = ax.get_ylim()
    ax.text(0, ylim[1] * 0.97, f"{event_label} Onset", color="black", ha="center", va="top", fontsize=8, fontweight="bold")

    for si, se in sig_runs:
        ax.hlines(y=ylim[1] * 0.9, xmin=t_rel[si], xmax=t_rel[se], linewidth=1.4, color="black")

    ax.set_xlabel(f"Time from {event_label.lower()} onset (s)")
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=11, fontweight="bold")

    sns.despine(ax=ax)
    plt.tight_layout()

    return fig, ax


In [ ]:
ts = session.trial_structure

for color_name, signal_type, label, main_color in [
    ("red", "spks", "Neurons", "#C62828"),
    ("green", "dff", "Astrocytes", "#2E7D32"),
]:
    tsd = session.tsd(color_name, signal_type)

    plot_eta(
        tsd.values, tsd.t, ts.shock_times, main_color,
        ylabel="Baseline-corrected\n(z-score)",
        title=f"{label} (n={tsd.shape[1]})",
    )
